# Server and protocol

The browser and Python talk in JSON messages: the browser proposes an edit, Python answers with an `ack` or a `reject`, and a run request gets the results back. This notebook sends those messages, first straight into the handler (no network), then over a real WebSocket to a server on a free local port. The editor page itself is not needed, so nothing has to be built first. If the machine blocks local sockets, the second half says so and stops, and the first half still ran.

In [1]:
# Standard library
import asyncio
import json
import tempfile
import threading
from pathlib import Path

In [2]:
# Third-party libraries
import uvicorn
from websockets.sync.client import connect

In [3]:
# Local libraries
import element_51
from element_51 import server
from element_51.edits import AddNode
from element_51.protocol import EditRequest

In [4]:
# A stand-in page, so the notebook does not need the built editor
PAGE = Path(tempfile.mkdtemp()) / "web"
PAGE.mkdir()
(PAGE / "index.html").write_text("<p>placeholder</p>", "utf-8")
server.WEB_DIR = PAGE
TIMEOUT = 10.0

## One node to talk about

In [5]:
@element_51.node
def number() -> float:
    return 21.0


app = element_51.App([number])
snapshot = json.loads(app.snapshot().model_dump_json())
print(sorted(snapshot), [t["name"] for t in snapshot["node_types"]])
assert snapshot["op"] == "snapshot"
assert snapshot["graph"]["nodes"] == []

['graph', 'jobs', 'node_types', 'op', 'theme', 'theme_changes', 'version'] ['number']


## Messages into the handler

`reply_to` takes the text of one message and returns Python's answer. A good edit is acknowledged and applied.

In [6]:
good = EditRequest(
    id=1, edit=AddNode(id="n1", node_type="number", x=10.0, y=20.0)
)
print(good.model_dump_json())
reply = await server.reply_to(app, good.model_dump_json())
print(type(reply).__name__, reply.model_dump_json())
assert type(reply).__name__ == "Ack"
assert [node.id for node in app.graph.nodes] == ["n1"]

{"op":"edit","id":1,"edit":{"kind":"add_node","id":"n1","node_type":"number","x":10.0,"y":20.0}}
Ack {"op":"ack","id":1}


The same id again is refused, with a reason, and the graph is unchanged.

In [7]:
again = await server.reply_to(app, good.model_dump_json())
print(type(again).__name__, again.model_dump_json())
assert type(again).__name__ == "Reject"
assert len(app.graph.nodes) == 1

Reject {"op":"reject","id":1,"reason":"node id n1 is already in use"}


A run request returns the values; a message that is not valid returns `None`.

In [8]:
ran = await server.reply_to(app, '{"op": "run"}')
print(type(ran).__name__)
assert type(ran).__name__ == "RunResult"
garbage = await server.reply_to(app, "not json")
print(garbage)
assert garbage is None

RunResult
None


## The same over a WebSocket

The server runs in a thread on port 0, so the system picks a free port.

In [9]:
wire_app = element_51.App([number])
config = uvicorn.Config(
    server.asgi_app(wire_app), host=server.HOST, port=0, log_level="warning"
)
instance = uvicorn.Server(config)
thread = threading.Thread(target=instance.run)
thread.start()
for _ in range(int(TIMEOUT * 100)):
    if instance.started:
        break
    await asyncio.sleep(0.01)
listening = instance.started
print("listening" if listening else "no local socket here, skipping")

listening


In [10]:
if listening:
    port = instance.servers[0].sockets[0].getsockname()[1]
    with connect(
        f"ws://{server.HOST}:{port}/ws", open_timeout=TIMEOUT
    ) as socket:
        first = json.loads(socket.recv(timeout=TIMEOUT))
        print("first message:", first["op"])
        assert first["op"] == "snapshot"
        socket.send(
            EditRequest(
                id=7, edit=AddNode(id="n1", node_type="number", x=1.0, y=2.0)
            ).model_dump_json()
        )
        answer = json.loads(socket.recv(timeout=TIMEOUT))
        print(answer)
        assert answer == {"op": "ack", "id": 7}
    assert len(wire_app.graph.nodes) == 1

first message: snapshot
{'op': 'ack', 'id': 7}


In [11]:
instance.should_exit = True
thread.join(timeout=TIMEOUT)
assert not thread.is_alive()